In [1]:
# ==========================================================
# Notebook 06: FAISS Candidate Retrieval
# ==========================================================

import pickle
import faiss
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
with open("processed/resume_embeddings.pkl", "rb") as file:

    resume_df = pickle.load(file)

resume_df.head()

,file_name,skills,experience,education,certifications,skills_embedding,experience_embedding,education_embedding,certification_embedding
0,10554236.pdf,skills accounting; general accounting; account...,experience company name july 2011 to november ...,education northern maine community college 199...,certifications certified defense financial man...,"[0.01941529, 0.0040915166, -0.045910846, 0.024...","[-0.06728348, 0.0024682328, -0.014528412, 0.02...","[-0.01684975, 0.076555185, -0.023776935, 0.031...","[-0.0033736788, -0.052531358, -0.08403263, 0.0..."
1,10674770.pdf,skills. highlights dba quick books mas - sage ...,experience staff accountant january 2014 to oc...,"education bachelor of science : accounting , m...",NaN,"[-0.043650076, -0.009040136, -0.03624853, 0.02...","[-0.0035784019, 0.0010006151, -0.029172646, 0....","[0.026238492, 0.012720538, 0.005227379, 0.0659...","[-0.06429787, -0.0282402, 0.045258433, -0.0351..."
2,11163645.pdf,skills and attributes. attributes self-motivat...,experience accountant january 2011 to november...,education computer applications specialist cer...,NaN,"[-0.023592636, -0.011933207, 0.0017404871, 0.0...","[-0.05706481, -0.011168787, -0.016337313, 0.05...","[-0.0080334665, 0.015025505, 0.018124653, 0.01...","[-0.06429787, -0.0282402, 0.045258433, -0.0351..."
3,11759079.pdf,skills by drafting over forty memorandums that...,experience company name june 2011 to current s...,"education emory university, goizueta business ...",certifications and awards fulton county casa b...,"[-0.004036917, -0.0071034115, -0.00849165, 0.0...","[0.03642456, -0.08111802, -0.11119465, 0.08057...","[-0.04872142, -0.004183779, -0.07013248, 0.011...","[-0.055658456, 0.023709623, -0.0633521, 0.0210..."
4,12065211.pdf,skills aderant/cms financial reporting excel u...,experience in full life cycle of general ledge...,education bachelor of business administration ...,"certifications certified public accountant, ne...","[-0.04393264, 0.0326399, -0.03382159, -0.00858...","[-0.009652888, 0.03468247, -0.021684052, -0.02...","[0.0060863015, 0.069854066, 0.007572601, -0.00...","[-0.017574374, -0.045755774, -0.01621892, -0.0..."


In [3]:
def build_resume_embedding(row):

    combined = (
        row["skills_embedding"]
        + row["experience_embedding"]
        + row["education_embedding"]
    ) / 3.0

    return combined

In [4]:
resume_vectors = []

for _, row in resume_df.iterrows():

    vector = build_resume_embedding(row)

    resume_vectors.append(vector)

resume_vectors = np.array(resume_vectors).astype("float32")

print(resume_vectors.shape)

(118, 384)


In [5]:
faiss.normalize_L2(resume_vectors)

In [6]:
embedding_dimension = resume_vectors.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)

print(index)

<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000001AA952FB9F0> >


In [7]:
index.add(resume_vectors)

print("Number of indexed resumes:", index.ntotal)

Number of indexed resumes: 118


In [9]:
faiss.write_index(index, "indexes/faiss_resume_index.bin")

print("FAISS index saved.")

FAISS index saved.


In [10]:
metadata = {"file_names": resume_df["file_name"].tolist()}

In [11]:
with open("indexes/" "resume_metadata.pkl", "wb") as file:

    pickle.dump(metadata, file)

print("Metadata saved.")

Metadata saved.


In [12]:
index = faiss.read_index("indexes/faiss_resume_index.bin")

with open("indexes/resume_metadata.pkl", "rb") as file:

    metadata = pickle.load(file)

print("Index loaded.")

Index loaded.


In [13]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 47f8e7b8-b514-4a41-bbab-685c918baa58)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 822b4063-ad1c-4e39-8107-b853683de9ae)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 3dfdfb05-306e-4983-a98e-21f1b0f0f477)')' thrown while requesting HEAD http

In [15]:
def build_jd_text(job_description):
    return f"""
    Skills:
    {job_description['skills']}

    Experience:
    {job_description['experience']}

    Education:
    {job_description['education']}
    """

In [16]:
job_description = {
    "skills": """
    Financial Accounting
    Bookkeeping
    Accounts Payable
    Accounts Receivable
    General Ledger
    Bank Reconciliation
    Financial Reporting
    Taxation
    GST
    TDS
    Payroll Processing
    Auditing
    Budgeting
    Microsoft Excel
    Tally ERP
    SAP FICO
    QuickBooks
    ERP Systems
    """,
    "experience": """
    Minimum 3 years experience
    in accounting, bookkeeping,
    financial reporting, taxation,
    and ERP-based financial systems.
    Experience with Tally, SAP,
    or QuickBooks is preferred.
    """,
    "education": """
    Bachelor's or Master's degree
    in Commerce, Accounting,
    Finance, or a related field.
    Professional certifications
    such as CA, CPA, CMA, or ACCA
    are an added advantage.
    """,
}

jd_text = build_jd_text(job_description)

jd_embedding = model.encode(jd_text)
jd_embedding = np.array([jd_embedding], dtype="float32")

faiss.normalize_L2(jd_embedding)

In [17]:
TOP_K = 5

scores, indices = index.search(jd_embedding, TOP_K)

In [18]:
print("Scores:", scores)

print("Indices:", indices)

Scores: [[0.7737995  0.7658733  0.76527154 0.761526   0.75505304]]
Indices: [[78 52 38 63 59]]


In [19]:
retrieved_candidates = []

for score, idx in zip(scores[0], indices[0]):

    retrieved_candidates.append(
        {"candidate": metadata["file_names"][idx], "similarity": round(float(score), 3)}
    )

In [20]:
leaderboard_df = pd.DataFrame(retrieved_candidates)

leaderboard_df

,candidate,similarity
0,28298773.pdf,0.774
1,23387174.pdf,0.766
2,20082776.pdf,0.765
3,25067742.pdf,0.762
4,24294778.pdf,0.755


In [23]:
import numpy as np
import pandas as pd
import faiss


def retrieve_top_candidates(
    job_description,
    model,
    index,
    metadata,
    top_k=5,
):

    # Convert JD dictionary to a single text
    jd_text = f"""
    Skills:
    {job_description['skills']}

    Experience:
    {job_description['experience']}

    Education:
    {job_description['education']}
    """

    # Generate embedding
    jd_embedding = model.encode(jd_text)

    # Convert to FAISS format
    jd_embedding = np.array([jd_embedding], dtype="float32")

    # Normalize
    faiss.normalize_L2(jd_embedding)

    # Search FAISS index
    scores, indices = index.search(
        jd_embedding,
        top_k,
    )

    # Collect results
    results = []

    for score, idx in zip(
        scores[0],
        indices[0],
    ):

        results.append(
            {
                "candidate": metadata["file_names"][idx],
                "score": round(float(score), 3),
            }
        )

    return pd.DataFrame(results)

In [24]:
retrieve_top_candidates(
    job_description=job_description,
    model=model,
    index=index,
    metadata=metadata,
    top_k=5,
)

,candidate,score
0,28298773.pdf,0.774
1,23387174.pdf,0.766
2,20082776.pdf,0.765
3,25067742.pdf,0.762
4,24294778.pdf,0.755
